# BettleBox 
The dataset from BLAZE paper consist of bug localization dataset from five different languages: Python, C++, Java, Javascript and Go

In [3]:
import pandas as pd
import json
import glob
from datasets import load_from_disk
import os
from collections import Counter
import ast

/home/cs21d002_eashaan/PhD/Objective1/obj1/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
# Path of the root directory of the BettleBox dataset
bettlebox_directory = "/home/user/CS21D002_A_Eashaan_Rao/Research/PhD/Objective1/Resources/BLAZE/15122980/Dataset/Dataset/BeetleBox"

In [5]:
# Step 1: Load all data files
# Load main bug beetlebox dataset
dataset = load_from_disk(bettlebox_directory)

# Combine train and test splits for a holistic overview
bettlebox_train = dataset['train'].to_pandas()
bettlebox_test = dataset['test'].to_pandas()

print(f"BettleBox Train: {bettlebox_train.shape}")
print(f"BettleBox Test: {bettlebox_test.shape}")
print(f"Columns: {bettlebox_train.columns}")

bettlebox_all = pd.concat([bettlebox_train, bettlebox_test], ignore_index=True)
print(f"Total records (Train + Test): {len(bettlebox_all)}")

BettleBox Train: (10191, 14)
BettleBox Test: (10445, 14)
Columns: Index(['status', 'repo_name', 'repo_url', 'issue_id', 'updated_files', 'title',
       'body', 'issue_url', 'pull_url', 'before_fix_sha', 'after_fix_sha',
       'report_datetime', 'language', 'commit_datetime'],
      dtype='object')
Total records (Train + Test): 20636


In [6]:
bettlebox_all.head(5)

,status,repo_name,repo_url,issue_id,updated_files,title,body,issue_url,pull_url,before_fix_sha,after_fix_sha,report_datetime,language,commit_datetime
0,closed,apache/airflow,https://github.com/apache/airflow,36219,"[""airflow/www/static/js/dag/details/taskInstan...","""Mark state as..."" button options grayed out",### Apache Airflow version\n\n2.7.3\n\n### If ...,https://github.com/apache/airflow/issues/36219,https://github.com/apache/airflow/pull/36254,a68b4194fe7201bba0544856b60c7d6724da60b3,20d547ecd886087cd89bcdf0015ce71dd0a12cef,2023-12-14 10:26:39+00:00,python,2023-12-16 14:25:25+00:00
1,closed,apache/airflow,https://github.com/apache/airflow,36187,"[""airflow/io/__init__.py"", ""tests/io/test_path...",Add unit tests to retrieve fsspec from provid...,### Body\n\nWe currently miss fsspec retrieval...,https://github.com/apache/airflow/issues/36187,https://github.com/apache/airflow/pull/36199,97e8f58673769d3c06bce397882375020a139cee,6c94ddf2bc123bfc7a59df4ce05f2b4e980f7a15,2023-12-12 15:58:07+00:00,python,2023-12-13 17:56:30+00:00
2,closed,apache/airflow,https://github.com/apache/airflow,36132,"[""airflow/providers/google/cloud/operators/clo...",Add overrides in the template field for the Go...,### Description\n\nThe overrides parameter is ...,https://github.com/apache/airflow/issues/36132,https://github.com/apache/airflow/pull/36133,df23df53155c7a3a9b30d206c962913d74ad3754,3dddfb4a4ae112544fd02e09a5633961fa725a36,2023-12-08 23:54:53+00:00,python,2023-12-11 15:27:29+00:00
3,closed,apache/airflow,https://github.com/apache/airflow,36102,"[""airflow/decorators/branch_external_python.py...",Using requirements file in VirtualEnvPythonOpe...,### Discussed in https://github.com/apache/air...,https://github.com/apache/airflow/issues/36102,https://github.com/apache/airflow/pull/36103,76d26f453000aa67f4e755c5e8f4ccc0eac7b5a4,3904206b69428525db31ff7813daa0322f7b83e8,2023-12-07 06:49:53+00:00,python,2023-12-07 09:19:54+00:00
4,closed,apache/airflow,https://github.com/apache/airflow,35949,"[""airflow/dag_processing/manager.py"", ""airflow...",dag processor deletes import errors of other d...,### Apache Airflow version\n\nmain (developmen...,https://github.com/apache/airflow/issues/35949,https://github.com/apache/airflow/pull/35956,9c1c9f450e289b40f94639db3f0686f592c8841e,1a3eeab76cdb6d0584452e3065aee103ad9ab641,2023-11-29 11:06:51+00:00,python,2023-11-30 13:29:52+00:00


In [39]:
# Convert stringified lists to actual Python lists
# bettlebox_all['updated_files'] = bettlebox_all['updated_files'].apply(ast.literal_eval)
print(bettlebox_all['updated_files'].apply(type).value_counts())
print(bettlebox_all['updated_files'].iloc[3])

updated_files
<class 'list'>    20636
Name: count, dtype: int64
['airflow/decorators/branch_external_python.py', 'airflow/decorators/branch_python.py', 'airflow/decorators/branch_virtualenv.py', 'airflow/decorators/external_python.py', 'airflow/decorators/python_virtualenv.py', 'airflow/decorators/short_circuit.py', 'airflow/models/abstractoperator.py', 'tests/decorators/test_branch_virtualenv.py', 'tests/decorators/test_external_python.py', 'tests/decorators/test_python_virtualenv.py']


In [15]:
# Step 2: Group By Language and Analyze Repos
languages = bettlebox_all['language'].unique()
print(f"Found {len(languages)} languages: {sorted(languages.tolist())}")

Found 5 languages: ['c++', 'go', 'java', 'javascript', 'python']


In [17]:
for lang in sorted(languages):
    print('-'*20)
    print(f"Analysis for Language: {lang}")
    lang_df = bettlebox_all[bettlebox_all['language'] == lang].copy()

    # Count total bug reports per repo
    bug_counts =  lang_df.groupby('repo_name')['issue_id'].count().reset_index(name='Total_Bug_Reports')

    # Count duplicate bug reports per repo
    duplicate_counts = lang_df.groupby('repo_name').apply(
        lambda x: x.shape[0] - x['body'].nunique()
    ).reset_index(name="Duplicate_Bug_Reports")

    # Combine and sort results
    results_df = pd.merge(bug_counts, duplicate_counts, on='repo_name')
    sorted_results = results_df.sort_values(by='Total_Bug_Reports', ascending=False).reset_index(drop=True)

    print(f"Found {len(sorted_results)} unique repositories for {lang}")
    print(sorted_results.to_string())


--------------------
Analysis for Language: c++
Found 5 unique repositories for c++
                  repo_name  Total_Bug_Reports  Duplicate_Bug_Reports
0         electron/electron               3032                   1491
1         godotengine/godot               1667                      0
2     ClickHouse/ClickHouse               1650                     22
3           bitcoin/bitcoin                598                      0
4  protocolbuffers/protobuf                235                      0
--------------------
Analysis for Language: go
Found 3 unique repositories for go
             repo_name  Total_Bug_Reports  Duplicate_Bug_Reports
0        dagger/dagger                451                      8
1  nats-io/nats-server                314                      2
2           nektos/act                226                      0
--------------------
Analysis for Language: java
Found 8 unique repositories for java
                 repo_name  Total_Bug_Reports  Duplicate_Bug_Reports

/tmp/ipykernel_63740/3129374963.py:10: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  duplicate_counts = lang_df.groupby('repo_name').apply(
/tmp/ipykernel_63740/3129374963.py:10: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  duplicate_counts = lang_df.groupby('repo_name').apply(
/tmp/ipykernel_63740/3129374963.py:10: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, 

## Extracting total bug reports, duplicate bug reports and total unique bug reports

In [4]:
# Group by repo and perform all aggregations using 'pull_url' for accuracy
repo_analysis = bettlebox_all.groupby('repo_name').agg(
    total_bug_reports=('issue_id', 'count'),
    unique_fix_count=('pull_url', 'nunique'),
    repo_link=('repo_url', 'first')
).reset_index()

# Calculate duplicate and unique bug report columns
repo_analysis['duplicate_bug_reports'] = repo_analysis['total_bug_reports'] - repo_analysis['unique_fix_count']
repo_analysis['total_unique_reports'] = repo_analysis['unique_fix_count']

# Rename 'repo_name' to 'repo' for consistency
repo_analysis.rename(columns={'repo_name': 'repo'}, inplace=True)

# Select and reorder final columns
final_columns = [
    'repo', 'repo_link', 'total_bug_reports', 'duplicate_bug_reports', 'total_unique_reports'
]

sorted_results = repo_analysis[final_columns].sort_values(
    by='total_bug_reports', ascending=False
).reset_index(drop=True)

print(sorted_results.to_string())


                        repo                                    repo_link  total_bug_reports  duplicate_bug_reports  total_unique_reports
0          electron/electron         https://github.com/electron/electron               3032                   1581                  1451
1          godotengine/godot         https://github.com/godotengine/godot               1667                    181                  1486
2      ClickHouse/ClickHouse     https://github.com/ClickHouse/ClickHouse               1650                    109                  1541
3    apache/dolphinscheduler   https://github.com/apache/dolphinscheduler               1393                     27                  1366
4             vercel/next.js            https://github.com/vercel/next.js               1377                    119                  1258
5             apache/airflow            https://github.com/apache/airflow               1212                     31                  1181
6            sveltejs/svelte      

In [5]:
sorted_results.to_csv("bettlebox_analysis_rd1.csv", index=False)

### Re-analyzing all repos to identify the type of ground truth files extension for all language

In [4]:
def get_file_extension(filepath):
    # Extracts the file extension from a given path.
    if '.' in filepath:
        return os.path.splitext(filepath)[1]
    return 'no_extension'

In [ ]:
def analyze_ground_truth_files_extension(bettlebox_df, lang):
    '''
    Analyzes the 'updates_files' for c++ repos in the beetlebox dataset.

    Args:
        bettlebox_df (pd.DataFrame): The combined DataFrame for the BeetleBox dataset
    '''
    print("Analyzing Ground Truth Files for C++ Projects in BeetleBox")

    # Filter for C++ language repos
    lang_df = bettlebox_df[bettlebox_df['language'] == lang].copy()

    if lang_df.empty:
        print(f"No {lang} projects found in the dataset.")
        return
    
    # The 'updated_files' column is a string representation of a list
    # We need to safely parse it into an actual list of strings. By using lambda with a try-except
    # block to handle potential errors.
    lang_df['updated_files_list'] = lang_df['updated_files'].apply(
        lambda x: ast.literal_eval(x) if isinstance(x, str) and x.startswith('[') else []
    )

    # Group by repository name
    repos = lang_df['repo_name'].unique()

    for repo in sorted(repos):
        print(f"Analysis for Repository: {repo}")
        repo_df = lang_df[lang_df['repo_name'] == repo]

        # Flatten the list of all updated files for this repo
        all_files_for_repo = [file for sublist in repo_df['updated_files_list'] for file in sublist]

        if not all_files_for_repo:
            print(" No updated files found for this repository.")
            continue

        # Get the extension for each file
        extensions = [get_file_extension(f) for f in all_files_for_repo]

        # Count the occurences of each extension
        extension_counts = Counter(extensions)
        print(extension_counts)

        print(" File type distribution (based on extension): ")
        # Sort by count in descending order
        for ext, count in sorted(extension_counts.items(), key=lambda item: item[1], reverse=True):
            print(f"    {ext:<15}: {count}")


In [17]:
analyze_ground_truth_files_extension(bettlebox_all, 'c++')

Analyzing Ground Truth Files for C++ Projects in BeetleBox
Analysis for Repository: ClickHouse/ClickHouse
 Unique File exensions across all c++ projects
['', '.ac', '.alpine', '.am', '.arm64', '.armv7', '.arrow', '.asar', '.avro', '.bash', '.bat', '.bazel', '.bin', '.bzl', '.c', '.cc', '.cfg', '.clj', '.cmake', '.conf', '.config', '.cpp', '.cs', '.csproj', '.css', '.dat', '.data', '.def', '.dict', '.expect', '.expected', '.fragment', '.gd', '.gemspec', '.glsl', '.gn', '.gni', '.go', '.grdp', '.gyp', '.gypi', '.h', '.hpp', '.html', '.ico', '.in', '.inc', '.include', '.init', '.install', '.j2', '.java', '.js', '.json', '.json5', '.kt', '.lib', '.limits', '.lock', '.m', '.m4', '.make', '.manifest', '.md', '.mjs', '.mk', '.mm', '.mp4', '.npy', '.out', '.patch', '.pb', '.pbxproj', '.php', '.plist', '.png', '.postinst', '.postinstall', '.preinst', '.prerm', '.props', '.proto', '.ps1', '.py', '.python', '.queries', '.rb', '.reference', '.scm', '.scss', '.sh', '.sigs', '.so', '.sql', '.supp', 

In [21]:
def analyze_ground_truth_unique_extension(bettlebox_df, lang):
    '''
    Finds a single unique list of all ground truth file extensons across all C++ projects in the
    BettleBox dataset.

    Args:
        bettlebox_df (pd.DataFrame): the combined dataframe for the BettleBox dataset
        lang: Primary language of the repository
    '''
    print(f" Finding Unique Ground Truth File Extensions for {lang} Projects")

    # Filter repos based on the language provided
    lang_df = bettlebox_df[bettlebox_df['language'] == lang].copy()

    if lang_df.empty:
        print(f"No {lang_df} projects found in the dataset.")
        return
    
    # The 'updated_files' column is a string representation of a list.
    # We need to safely parse it into an actual list of strings.
    lang_df['updated_files_list'] = lang_df['updated_files'].apply(
        lambda x: ast.literal_eval(x) if isinstance(x, str) and x.startswith('[') else []
    )

    # Flatten the list of all updated files from all C++ repos into a single list
    all_lang_files = [
        file for sublist in lang_df['updated_files_list'] for file in sublist
    ]

    # Get the unique extension for each file using a set for automatic uniqueness
    unique_extensions = {get_file_extension(f) for f in all_lang_files}

    print(f"Unique File Extension across all {lang} projects")
    print(sorted(list(unique_extensions)))
    

In [19]:
analyze_ground_truth_unique_extension(bettlebox_all, 'c++')

 Finding Unique Ground Truth File Extensions for C++ Projects
Unique File Extension across all c++ projects
['', '.ac', '.alpine', '.am', '.arm64', '.armv7', '.arrow', '.asar', '.avro', '.bash', '.bat', '.bazel', '.bin', '.bzl', '.c', '.cc', '.cfg', '.clj', '.cmake', '.conf', '.config', '.cpp', '.cs', '.csproj', '.css', '.dat', '.data', '.def', '.dict', '.expect', '.expected', '.fragment', '.gd', '.gemspec', '.glsl', '.gn', '.gni', '.go', '.grdp', '.gyp', '.gypi', '.h', '.hpp', '.html', '.ico', '.in', '.inc', '.include', '.init', '.install', '.j2', '.java', '.js', '.json', '.json5', '.kt', '.lib', '.limits', '.lock', '.m', '.m4', '.make', '.manifest', '.md', '.mjs', '.mk', '.mm', '.mp4', '.npy', '.out', '.patch', '.pb', '.pbxproj', '.php', '.plist', '.png', '.postinst', '.postinstall', '.preinst', '.prerm', '.props', '.proto', '.ps1', '.py', '.python', '.queries', '.rb', '.reference', '.scm', '.scss', '.sh', '.sigs', '.so', '.sql', '.supp', '.svg', '.targets', '.templates', '.ts', '.tx

In [20]:
analyze_ground_truth_unique_extension(bettlebox_all, 'go')

 Finding Unique Ground Truth File Extensions for C++ Projects
Unique File Extension across all go projects
['', '.Dockerfile', '.bash', '.bats', '.conf', '.css', '.cue', '.ex', '.exs', '.fragment', '.go', '.graphqls', '.gtpl', '.hbs', '.ignore', '.js', '.json', '.key', '.lock', '.md', '.mdx', '.mjs', '.mod', '.mts', '.nightly', '.pem', '.png', '.py', '.rs', '.scss', '.sh', '.sum', '.tmpl', '.toml', '.ts', '.txt', '.win64', '.xml', '.yaml', '.yml', 'no_extension']


In [22]:
analyze_ground_truth_unique_extension(bettlebox_all, 'java')

 Finding Unique Ground Truth File Extensions for java Projects
Unique File Extension across all java projects
['', '.ClusterFilter', '.CommandStep', '.Executor', '.Filter', '.LiquibaseDataType', '.MeshEnvListenerFactory', '.Protocol', '.SnapshotGenerator', '.SqlGenerator', '.TypeBuilder', '.Validation', '.asciidoc', '.bat', '.bazel', '.bz2', '.bzl', '.cer', '.cmd', '.conf', '.config', '.cpp', '.crt', '.crx', '.cs', '.css', '.csv', '.csv-spec', '.db', '.desktop', '.enc', '.factories', '.fxml', '.gemspec', '.gradle', '.groovy', '.gz', '.h', '.html', '.importorder', '.ini', '.install4j', '.iss', '.jar', '.java', '.jj', '.jpg', '.js', '.json', '.key', '.less', '.lock', '.md', '.mustache', '.nuspec', '.ods', '.plist', '.png', '.policy', '.properties', '.ps1', '.py', '.rb', '.rs', '.rst', '.scss', '.sh', '.sha1', '.sql', '.sql-spec', '.st', '.svg', '.targets', '.toml', '.tpl', '.ts', '.tsv', '.tsx', '.txt', '.vm', '.vt', '.vue', '.wxi', '.wxs', '.xls', '.xml', '.xsd', '.yaml', '.yml', '.zip'

In [23]:
analyze_ground_truth_unique_extension(bettlebox_all, 'javascript')

 Finding Unique Ground Truth File Extensions for javascript Projects
Unique File Extension across all javascript projects
['', '.Dockerfile', '.avif', '.cjs', '.coffee', '.css', '.cts', '.example', '.graphql', '.graphqls', '.html', '.ico', '.jpg', '.js', '.jsm', '.json', '.jsx', '.link', '.lock', '.map', '.md', '.mdx', '.mjs', '.mts', '.opts', '.pdf', '.png', '.properties', '.rb', '.rs', '.scss', '.sh', '.snap', '.sqlite', '.stderr', '.svelte', '.svg', '.toml', '.ts', '.tsx', '.ttf', '.txt', '.wasm', '.webp', '.xml', '.yaml', '.yml', 'no_extension']


In [24]:
analyze_ground_truth_unique_extension(bettlebox_all, 'python')

 Finding Unique Ground Truth File Extensions for python Projects
Unique File Extension across all python projects
['', '.0', '.1', '.Dockerfile', '.acl', '.ambr', '.base', '.bat', '.cc', '.cfg', '.ci', '.cmake', '.cnf', '.conf', '.cs', '.csproj', '.css', '.csv', '.db', '.example', '.expected', '.g4', '.gif', '.h', '.html', '.in', '.ini', '.interp', '.inv', '.inventory', '.ipynb', '.j2', '.jar', '.java', '.jinja2', '.js', '.json', '.json5', '.jsx', '.kubernetes-helm-yaml', '.less', '.lock', '.manifest', '.md', '.mdx', '.mkv', '.mp4', '.nodejs14x', '.onnx', '.pb', '.php', '.png', '.pot', '.proto', '.proto3', '.ps1', '.psm1', '.py', '.pyi', '.rdb', '.rst', '.scss', '.sh', '.sha256', '.sln', '.stderr', '.stdout', '.svg', '.tf', '.tgz', '.toml', '.ts', '.tsx', '.txt', '.xml', '.yaml', '.yml', '.zip', 'no_extension']


# MetaData Extraction
Set of methods to calculate all ten metadata for selected projects from BettleBox

1. LOC
2. age_years
3. median_bug_year
4. num_authors
5. num_commits
6. num_dependencies
7. polyglot_index
8. bug_density
9. code_complexity
10. bug_report_verbosity

In [21]:
# Import libraries
import pandas as pd
import os
import re
import git
import subprocess
from datetime import datetime
from tqdm import tqdm
import xml.etree.ElementTree as ET
import tempfile 
from datasets import load_from_disk

## Configuration

In [22]:
# 1. Path to the input CSV file
# Must have columns: 'repo_name', 'language', 'total_unique_bug_report'
INPUT_CSV_PATH = "/home/user/CS21D002_A_Eashaan_Rao/Research/PhD/Objective1/benchmark_dataset_analysis/bettlebox_metadata_input.csv"

# 2. Path to the main BettleBox dataset file (to get commite and date info)
# Path of the root directory of the BettleBox dataset
bettlebox_directory = "/home/user/CS21D002_A_Eashaan_Rao/Research/PhD/Objective1/Resources/BLAZE/15122980/Dataset/Dataset/BeetleBox"

# Load main bug beetlebox dataset
dataset = load_from_disk(bettlebox_directory)

# Combine train and test splits for a holistic overview
bettlebox_train = dataset['train'].to_pandas()
bettlebox_test = dataset['test'].to_pandas()

bettlebox_df = pd.concat([bettlebox_train, bettlebox_test], ignore_index=True)
bettlebox_df['bug_report'] = bettlebox_df['title'] + "\n" + bettlebox_df['body']

# 3. Path to the parent directory where repos are cloned
CLONE_DIR = '/home/user/CS21D002_A_Eashaan_Rao/Research/PhD/Objective1/temp_repos/'

# 4. Path for the final output CqSV file.
OUTPUT_CSV_PATH = '/home/user/CS21D002_A_Eashaan_Rao/Research/PhD/Objective1/benchmark_dataset_analysis/bettlebox_metadata.csv'

# 5. Extensions for Polyglot Index Calculation
LANGUAGE_EXTENSIONS = {
    'c++': ['.c', '.cc', '.cmake', '.cpp', '.cxx', '.h', '.hh', '.hpp', '.hxx', '.in', '.json', '.make', '.py', '.sh', '.xml'],
    'go': ['.go', '.json', '.proto', '.sh', '.yaml', '.yml'],
    'java': ['.gradle', '.groovy', '.java', '.json', '.properties', '.xml', '.yml', '.yaml'],
    'javascript': ['.css', '.html', '.js', '.json', '.jsx', '.mjs', '.scss', '.sh', '.ts', '.tsx', '.yaml', '.yml'],
    'kotlin': ['.gradle', '.json', '.kt', '.kts', '.properties', '.xml', '.yaml', '.yml'],
    'python': ['.bash', '.cfg', '.in', '.ini', '.json', '.py', '.sh', '.toml', '.yaml', '.yml']
}
PRIMARY_EXTENSIONS = {
    'python': ['.py'],
    'java': ['.java'],
    'kotlin': ['.kt'],
    'c++': ['.c', '.cc', '.cpp', '.cxx', '.h', '.hh', '.hpp', '.hxx'],
    'go': ['.go'],
    'javascript': ['.js', '.jsx', '.mjs', '.ts', '.tsx']
}

## Helper Methods

In [23]:
# Pre-requisite: Assume 'beetlebox_df' is loaded and has the 'report_datetime' column
print(" Verifying the corrected 'report_datetime' column")

# The 'report_datetime' column should already be in datetime formate for your pre-processing
# we will just ensure it, coercing any potential errors that might still exist
bettlebox_df['report_datetime'] = pd.to_datetime(bettlebox_df['report_datetime'])

# 1. Show basic statistics of the corrected dates
print("Overall corrected date statistics:")
# Filter out any NaT values for accurate stats
valid_dates = bettlebox_df['report_datetime'].dropna()
if not valid_dates.empty:
    print(f" Earliest Date: {valid_dates.min()}")
    print(f" Latest Date: {valid_dates.max()}")
    print(f" Median Date: {valid_dates.median()}")
else:
    print(" No valid dates found in the 'report_datetime' column.")

# 2. Show the distribution of bug reports by year
print("Distribution of Bug Reports per Year (top 15):")
print(valid_dates.dt.year.value_counts().sort_index(ascending=False).head(15).to_string())

# 3. Isolate and check if any '1970' dates remain
print("Checking for any remaining anomalous '1970' dates")
problematic_rows = bettlebox_df[bettlebox_df['report_datetime'].dt.year == 1970]

if not problematic_rows.empty:
    print(f"Warning: Found {len(problematic_rows)} entries that still have a '1970' year")
else:
    print("Success: No entries with a '1970' year were found in the 'report_datetime' column.")

 Verifying the corrected 'report_datetime' column
Overall corrected date statistics:
 Earliest Date: 2010-12-19 16:22:54+00:00
 Latest Date: 2024-03-07 17:19:02+00:00
 Median Date: 2021-10-22 18:26:06+00:00
Distribution of Bug Reports per Year (top 15):
report_datetime
2024     271
2023    5015
2022    4238
2021    3324
2020    2553
2019    1866
2018    1122
2017    1013
2016     845
2015     155
2014      36
2013      44
2012     142
2011      11
2010       1
Checking for any remaining anomalous '1970' dates
Success: No entries with a '1970' year were found in the 'report_datetime' column.


In [24]:
def count_lines_in_file(file_path):
    '''
    Counts lines in a file, handling encoding errors.
    '''
    try:
        with open(file_path, 'r', encoding='utf-8', errors='ignore') as f:
            return len(f.readlines())
    except Exception:
        return 0

In [25]:
def calculate_loc_and_polyglot(repo_path, declared_language):
    '''
    Calculates LoC and polyglot index. Auto-detects the actual primary language in the snapshot to handle
    migrations
    '''
    loc_by_lang = {lang: 0 for lang in PRIMARY_EXTENSIONS.keys()}
    loc_relevant = 0

    # first, calculate LoC for each potential primary language
    for root, _, files in os.walk(repo_path):
        for file in files:
            for lang, exts in PRIMARY_EXTENSIONS.items():
                if file.endswith(tuple(exts)):
                    loc_by_lang[lang] += count_lines_in_file(os.path.join(root, file))

    # Auto-detect the language with the most LoC in this snapshot
    actual_primary_languge = max(loc_by_lang, key=loc_by_lang.get) if loc_by_lang else declared_language
    loc_primary = loc_by_lang.get(actual_primary_languge, 0)

    # Now, calculate total relevant LoC based on the DECLARED ecosystem
    relevant_exts = tuple(LANGUAGE_EXTENSIONS.get(declared_language.lower(), []))

    for root, _, files in os.walk(repo_path):
        for file in files:
            if file.endswith(relevant_exts):
                loc_relevant += count_lines_in_file(os.path.join(root, file))
                
    polyglot_index = (loc_primary / loc_relevant) if loc_relevant > 0 else 0
    return loc_relevant, polyglot_index

In [26]:
def count_dependencies(repo_path, language):
    """
    Calculates a proxy for dependency complexity by summing the Lines of Code (LoC)
    of standard dependency files for the primary language.
    """
    loc_count = 0
    lang = language.lower()

    dependency_files = []
    if lang == 'python':
        dependency_files = ['requirements.txt', 'pyproject.toml']
    elif lang in ['java', 'kotlin']:
        dependency_files = ['pom.xml', 'build.gradle', 'build.gradle.kts']
    elif lang == 'c++':
        dependency_files = ['CMakeLists.txt', 'Makefile'] # Example for C++
    elif lang == 'javascript':
        dependency_files = ['package.json'] # Example for JS
    elif lang == 'go':
        dependency_files = ['go.mod'] # Example for Go

    for root, _, files in os.walk(repo_path):
        for file in files:
            if file in dependency_files:
                file_path = os.path.join(root, file)
                # Simply add the number of lines in the file to the count
                loc_count += count_lines_in_file(file_path)
    
    return loc_count

In [27]:
def calculate_average_complexity(repo_path, language):
    '''
    Calculates average cyclomatic complexity using the 'lizard' tool.
    We store the ouput of lizard to a temporary file to handle large outputs reliably.
    '''
    # Create a temp file to stroe the lizard output
    with tempfile.NamedTemporaryFile(mode='w+', delete=False, suffix='.txt', encoding='utf-8') as temp_out:
        temp_filename = temp_out.name

    try:
        # lizard expects 'c++' to be written as 'cpp'
        lang_for_lizard = 'cpp' if language.lower() == 'c++' else language.lower()
        # print("Language for lizard: ", lang_for_lizard)

        # Redirect stdout to the temp file
        # We run the command and tell it to write its output directly to our temp file
        result = subprocess.run(
            ['lizard', '-i', '0', repo_path], # another command ['lizard', '-l', 'lang_for_lizard', repo_path]
            stdout = open(temp_filename, 'w',  encoding='utf-8'), # Write stdout to the temp file
            stderr=subprocess.PIPE, # Still capture any errors in memory 
            check=False, text=True
        )

        # We can still manually check the return code if we want to log detailed errors
        if result.returncode != 0:
            print(f"\nWarning: Lizard finished with a non-zero exit code ({result.returncode}) for {repo_path}. This usually indicates warnings were found. Continuing to parse output.")
            # We don't return here, because the output file is likely still valid.

        # Read the results back from the file
        with open(temp_filename, 'r', encoding='utf-8') as f:
            lizard_output = f.read()

        # Get all non-empty lines from the output
        lines = [line for line in lizard_output.strip().splitlines()]

        # the summary data is on the second to last line
        if len(lines) >=3 :
            # target the line with the numbers (the last non-empty line)
            summary_line = lines[-1]

            # Split the line by whitespace
            values = summary_line.split()

            if len(values) >= 3:
                # The Avg CCN is the 3rd value (index 2)
                avg_ccn = float(values[2])
                return avg_ccn

        # Find the summary line in the output
        # If we reach here, the summary line was not found or was malformed
        print(f"\nWarning: Could not parse lizard summary for {repo_path}.")
        return 0.0
        
    except FileNotFoundError:
        # This error is critical, so we print it once and then it will return 0 for others.
        print("\nERROR: 'lizard' command not found. Please install it with 'pip install lizard'.")
        return 0.0
    except subprocess.CalledProcessError as e:
        print(f"\n Lizard command failed for {repo_path}. Stderr: {e.stderr}")
        return 0.0
    except (IndexError, ValueError) as e:
        print(f"\nFailed to extract complexity value from summary line for {repo_path}. Error: {e}")
        return 0.0
    except Exception as e:
        print(f"\nAn unexpected error occurred in calculate_average_complexity: {e}")
        return 0.0
    finally:
        if os.path.exists(temp_filename):
            os.remove(temp_filename)

## Main Code

In [28]:
print("Starting metadata extraction for BeetleBox Dataset...")

# Load input files
try:
    selected_repos_df = pd.read_csv(INPUT_CSV_PATH)
    # print(selected_repos_df)
except FileNotFoundError as e:
    print(f"Error: Input file not found. {e}")
    exit(0)


results = []

for _, row in tqdm(selected_repos_df.iterrows(), total=len(selected_repos_df), desc="Processing Repos"):
    repo_name = row['repo_name']
    language = row['language']
    unique_bugs = row['total_unique_bug_report']

    repo_path = os.path.join(CLONE_DIR, language.lower(), repo_name.replace('/', '_'))
    print(f"Calculating meta data for repo: {repo_name}")
    if not os.path.exists(repo_path):
        print(f"Warning: Clone repo not found for {repo_name} at {repo_path}. Skipping.")
        continue

    repo_bettlebox_data = bettlebox_df[bettlebox_df['repo_name'] == repo_name].copy()
    if repo_bettlebox_data.empty:
        print(f"Warning: No data found for {repo_name} in LCA main file. Skipping.")
        continue

    # Get the snapshot commit from the latest bug report
    latest_bug = repo_bettlebox_data.sort_values(by='report_datetime', ascending=False).iloc[0]
    snapshot_commit = latest_bug['before_fix_sha']

    try:
        repo = git.Repo(repo_path)
        repo.git.checkout(snapshot_commit, f=True)

        # *****************
        # Calculate Metrics
        # *****************

        # c) Age
        # --- Metrics now use the reliable 'report_date' column ---
        min_date = repo_bettlebox_data['report_datetime'].min()
        max_date = repo_bettlebox_data['report_datetime'].max()
        age_years = ((max_date - min_date).days) / 365.25
        median_bug_year = repo_bettlebox_data['report_datetime'].dt.year.median()

        # d) No. of authors & e) No. of commits (Correctly scoped to the snapshot)
        all_commits = list(repo.iter_commits())
        num_commits = len(all_commits)
        num_authors = len({c.author.email for c in all_commits})

        # a) LoC & g) Polyglot Index
        loc, polyglot_index = calculate_loc_and_polyglot(repo_path, language)

        # f) No. of external dependencies
        dependencies = count_dependencies(repo_path, language)

        # h) Bug density
        kloc = loc / 1000
        bug_density = (kloc / unique_bugs) if unique_bugs > 0 else 0

        # i) code_complexity
        code_complexity = calculate_average_complexity(repo_path, language)

        # j) Calculate bug report verbosoty
        bug_report_verbosity = repo_bettlebox_data['bug_report'].str.split().str.len().mean()
        
        results.append({
            'repo_name': repo_name,
            'language': language,
            'LoC': loc,
            'age_years': age_years,
            'median_bug_year': median_bug_year, # Raw data for later categorization
            'num_authors': num_authors,
            'num_commits': num_commits,
            'num_dependencies': dependencies,
            'polyglot_index': polyglot_index,
            'bug_density': bug_density,
            'code_complexity': code_complexity,
            'bug_report_verbosity': bug_report_verbosity
        })
    
    # except git.exec.GitCommandError as e:
    #     print(f"Error processing Git Repo {repo_name}: {e}")
    except Exception as e:
        print(f"An unexpected error occurred for {repo_name}: {e}")

if not results:
    print("No results were generated.")
    exit(0)

# Create final DataFrame
final_df = pd.DataFrame(results)
# print(final_df.to_string())

# # b) Calculate project_size category
# final_df = final_df.groupby('language', group_keys=False).apply(categorize_by_tercile)

# Save to CSV
final_df.to_csv(OUTPUT_CSV_PATH, index=False)
print(f"Metdata extraction complete. Results saved to '{OUTPUT_CSV_PATH}'")


Starting metadata extraction for BeetleBox Dataset...


Processing Repos:   0%|          | 0/1 [00:00<?, ?it/s]

Calculating meta data for repo: protocolbuffers/protobuf


Processing Repos: 100%|██████████| 1/1 [00:20<00:00, 20.13s/it]


                  repo_name language     LoC  age_years  median_bug_year  num_authors  num_commits  num_dependencies  polyglot_index  bug_density  code_complexity  bug_report_verbosity
0  protocolbuffers/protobuf      C++  516317   8.342231           2019.0         1228        16554               744        0.915463     2.254659              2.3            188.319149


## Polglot Analysis

In [2]:
bettlebox_directory = "/home/cs21d002_eashaan/PhD/Objective1/Resources/BLAZE/15122980/Dataset/Dataset/BeetleBox"
# Step 1: Load all data files
# Load main bug beetlebox dataset
dataset = load_from_disk(bettlebox_directory)

# Combine train and test splits for a holistic overview
bettlebox_train = dataset['train'].to_pandas()
bettlebox_test = dataset['test'].to_pandas()

print(f"BettleBox Train: {bettlebox_train.shape}")
print(f"BettleBox Test: {bettlebox_test.shape}")
print(f"Columns: {bettlebox_train.columns}")

bettlebox_all = pd.concat([bettlebox_train, bettlebox_test], ignore_index=True)
print(f"Total records (Train + Test): {len(bettlebox_all)}")

BettleBox Train: (10191, 14)
BettleBox Test: (10445, 14)
Columns: Index(['status', 'repo_name', 'repo_url', 'issue_id', 'updated_files', 'title',
       'body', 'issue_url', 'pull_url', 'before_fix_sha', 'after_fix_sha',
       'report_datetime', 'language', 'commit_datetime'],
      dtype='object')
Total records (Train + Test): 20636


In [3]:
import pandas as pd

# Helper function to extract extensions from a list of filenames
def extract_extensions(file_list):
    if isinstance(file_list, list):
        return set(f.split('.')[-1] for f in file_list if '.' in f)
    return set()

# Apply extension extraction to each row
bettlebox_all['file_exts'] = bettlebox_all['updated_files'].apply(extract_extensions)

# Flag rows where multiple extensions are present
bettlebox_all['multi_ext_flag'] = bettlebox_all['file_exts'].apply(lambda x: len(x) > 1)

# Group by language
lang_groups = bettlebox_all.groupby('language')

# Analyze each language group
for lang, lang_df in lang_groups:
    print(f"\n🌐 Language: {lang}")
    repo_groups = lang_df.groupby('repo_name')
    
    for repo, repo_df in repo_groups:
        multi_ext_ids = repo_df[repo_df['multi_ext_flag']]['issue_id'].tolist()
        unique_exts = set().union(*repo_df['file_exts'].tolist())
        
        print(f"🔧 Repo: {repo}")
        print(f"🧮 Instances with multiple extensions: {len(multi_ext_ids)}")
        print(f"📄 Issue IDs with multiple extensions: {multi_ext_ids}")
        print(f"🧷 Unique extensions in updated files: {sorted(unique_exts)}")



🌐 Language: c++
🔧 Repo: ClickHouse/ClickHouse
🧮 Instances with multiple extensions: 0
📄 Issue IDs with multiple extensions: []
🧷 Unique extensions in updated files: []
🔧 Repo: bitcoin/bitcoin
🧮 Instances with multiple extensions: 0
📄 Issue IDs with multiple extensions: []
🧷 Unique extensions in updated files: []
🔧 Repo: electron/electron
🧮 Instances with multiple extensions: 0
📄 Issue IDs with multiple extensions: []
🧷 Unique extensions in updated files: []
🔧 Repo: godotengine/godot
🧮 Instances with multiple extensions: 0
📄 Issue IDs with multiple extensions: []
🧷 Unique extensions in updated files: []
🔧 Repo: protocolbuffers/protobuf
🧮 Instances with multiple extensions: 0
📄 Issue IDs with multiple extensions: []
🧷 Unique extensions in updated files: []

🌐 Language: go
🔧 Repo: dagger/dagger
🧮 Instances with multiple extensions: 0
📄 Issue IDs with multiple extensions: []
🧷 Unique extensions in updated files: []
🔧 Repo: nats-io/nats-server
🧮 Instances with multiple extensions: 0
📄 Issu

In [1]:
# Helper function to extract extensions from a list of filenames
def extract_extensions(file_entry):
    # Convert to list if it's a string or numpy array-like
    if isinstance(file_entry, str):
        # Handle stringified list: remove brackets and split
        file_list = file_entry.strip("[]").replace("'", "").split()
    elif hasattr(file_entry, '__iter__'):
        file_list = list(file_entry)
    else:
        return set()
    
    # Extract extensions
    return set(f.split('.')[-1] for f in file_list if '.' in f)


In [10]:
# Load the Parquet file
df = pd.read_parquet('/home/cs21d002_eashaan/PhD/Objective1/data/processed/bug_reports_clean.parquet')
print(df.columns)
# # Filter rows where source_dataset is 'bettlebox'
# bettlebox_projects = df[df['source_dataset'] == 'BeetleBox']['repo_name'].unique()

# # Display the project names
# print("📦 Projects from 'bettlebox' dataset:")
# print(bettlebox_projects)


Index(['repo_name', 'bug_id', 'bug_report_text', 'ground_truth_files',
       'creation_date', 'fix_date', 'pre_fix_commit_sha', 'fix_commit_sha',
       'language', 'source_dataset', 'bug_report_url', 'fix_url'],
      dtype='object')


In [9]:
# Filter for BeetleBox dataset
beetlebox_df = df[df['source_dataset'] == 'BeetleBox'].copy()

# Extract extensions per row
beetlebox_df['file_exts'] = beetlebox_df['ground_truth_files'].apply(extract_extensions)

# Flag rows with multiple extensions
beetlebox_df['multi_ext_flag'] = beetlebox_df['file_exts'].apply(lambda x: len(x) > 1)

# Group by repo_name
grouped = beetlebox_df.groupby('repo_name')

# Analyze each repo
for repo, group in grouped:
    total_bugs = group['bug_id'].nunique()
    multi_ext_bugs = group[group['multi_ext_flag']]['bug_id'].tolist()
    single_ext_bugs = total_bugs - len(multi_ext_bugs)
    unique_exts = set().union(*group['file_exts'].tolist())

    # if multi_ext_bugs:  # Only show repos with multiple extensions
    #     unique_exts = set().union(*group['file_exts'].tolist())
        
    print(f"\n📁 Repo: {repo}")
    print(f"🐞 Total bug reports: {total_bugs}")
    print(f"🔀 Bug reports with multiple extensions: {len(multi_ext_bugs)}")
    print(f"🔧 Bug reports with single extension: {single_ext_bugs}")
    print(f"🆔 Bug IDs with multiple extensions: {multi_ext_bugs}")
    print(f"🧷 Unique extensions in repo: {sorted(unique_exts)}")


📁 Repo: ansible/ansible
🐞 Total bug reports: 768
🔀 Bug reports with multiple extensions: 506
🔧 Bug reports with single extension: 262
🆔 Bug IDs with multiple extensions: ['82359', '82353', '82264', '82244', '82241', '82226', '82179', '82142', '82024', '82020', '82018', '81901', '81710', '81666', '81656', '81574', '81553', '81533', '81532', '81474', '81404', '81188', '81163', '81053', '80880', '80863', '80853', '80835', '80709', '80605', '80590', '80561', '80523', '80506', '80478', '80427', '80422', '80420', '80418', '80417', '80415', '80413', '80411', '80410', '80408', '80303', '80256', '80128', '80110', '80089', '79968', '79956', '79942', '79862', '79836', '79833', '79763', '79749', '79683', '79680', '79676', '79577', '79463', '79411', '79368', '79101', '79083', '79023', '78932', '78882', '78795', '78793', '78762', '78693', '78675', '78612', '78611', '78509', '78492', '78490', '78442', '78438', '78348', '78295', '78288', '78283', '78156', '78141', '78131', '78112', '78042', '77928', 

## Calculating types of bug reports

In [11]:
import pandas as pd

In [19]:
LANG_TO_EXT = {
    'python': 'py',
    'java': 'java',
    'c++': 'cpp',
    'go': 'go',
    'javascript': 'js'
}

In [ ]:
# Extract extensions from 
def extract_extensions(file_entry):
    if isinstance(file_entry, str):
        file_list = file_entry.strip("[]").replace("'", "").split(',')
        file_list = [f.strip() for f in file_list]
    elif hasattr(file_entry, '__iter__'):
        file_list = list(file_entry)
    else:
        return set()
    return set(f.split('.')[-1] for f in file_list if '.' in f)

In [14]:
# --- Load and filter BeetleBox dataset ---
df = pd.read_parquet('/home/cs21d002_eashaan/PhD/Objective1/data/processed/bug_reports_clean.parquet')
beetlebox_df = df[df['source_dataset'] == 'BeetleBox'].copy()

In [20]:
# --- Extract extensions and primary language ---
beetlebox_df['file_exts'] = beetlebox_df['ground_truth_files'].apply(extract_extensions)
beetlebox_df['primary_lang'] = beetlebox_df['language'].str.lower().map(LANG_TO_EXT)

In [21]:
# --- Define classification flags ---
def classify_bug(exts, primary_lang):
    if not exts:
        return 'invalid'
    if len(exts) == 1 and primary_lang in exts:
        return 'mono_fix'
    elif len(exts) >= 2 and primary_lang in exts:
        return 'poly_fix'
    else:
        return 'other'

In [22]:
beetlebox_df['fix_type'] = beetlebox_df.apply(lambda row: classify_bug(row['file_exts'], row['primary_lang']), axis=1)

In [24]:
# --- Mono-fix-all simulation: strip non-primary extensions ---
def mono_fix_all(exts, primary_lang):
    return set([ext for ext in exts if ext == primary_lang])

beetlebox_df['mono_fix_all_exts'] = beetlebox_df.apply(lambda row: mono_fix_all(row['file_exts'], row['primary_lang']), axis=1)
grouped = beetlebox_df.groupby('repo_name')

# --- Count categories ---
# total_overall = beetlebox_df['bug_id'].nunique()
# mono_fix_count = beetlebox_df[beetlebox_df['fix_type'] == 'mono_fix']['bug_id'].nunique()
# poly_fix_count = beetlebox_df[beetlebox_df['fix_type'] == 'poly_fix']['bug_id'].nunique()
# mono_fix_all_count = beetlebox_df[beetlebox_df['mono_fix_all_exts'].apply(lambda x: len(x) > 0)]['bug_id'].nunique()

for repo, group in grouped:
    total_overall = group['bug_id'].nunique()
    mono_fix_count = group[group['fix_type'] == 'mono_fix']['bug_id'].nunique()
    poly_fix_count = group[group['fix_type'] == 'poly_fix']['bug_id'].nunique()
    mono_fix_all_count = group[group['mono_fix_all_exts'].apply(lambda x: len(x) > 0)]['bug_id'].nunique()

    print(f"\n📁 Repo: {repo}")
    print(f"🟢 Overall bug reports: {total_overall}")
    print(f"🔵 Mono-fix-all simulated bug reports: {mono_fix_all_count}")
    print(f"🟣 Mono-fix bug reports: {mono_fix_count}")
    print(f"🟠 Poly-fix bug reports: {poly_fix_count}")



📁 Repo: ansible/ansible
🟢 Overall bug reports: 768
🔵 Mono-fix-all simulated bug reports: 627
🟣 Mono-fix bug reports: 150
🟠 Poly-fix bug reports: 477

📁 Repo: apache/airflow
🟢 Overall bug reports: 1181
🔵 Mono-fix-all simulated bug reports: 917
🟣 Mono-fix bug reports: 603
🟠 Poly-fix bug reports: 314

📁 Repo: apache/dolphinscheduler
🟢 Overall bug reports: 1363
🔵 Mono-fix-all simulated bug reports: 698
🟣 Mono-fix bug reports: 501
🟠 Poly-fix bug reports: 197

📁 Repo: apache/dubbo
🟢 Overall bug reports: 379
🔵 Mono-fix-all simulated bug reports: 253
🟣 Mono-fix bug reports: 226
🟠 Poly-fix bug reports: 27

📁 Repo: axios/axios
🟢 Overall bug reports: 113
🔵 Mono-fix-all simulated bug reports: 78
🟣 Mono-fix bug reports: 54
🟠 Poly-fix bug reports: 24

📁 Repo: bitcoin/bitcoin
🟢 Overall bug reports: 563
🔵 Mono-fix-all simulated bug reports: 345
🟣 Mono-fix bug reports: 132
🟠 Poly-fix bug reports: 213

📁 Repo: ccxt/ccxt
🟢 Overall bug reports: 690
🔵 Mono-fix-all simulated bug reports: 40
🟣 Mono-fix bug 

In [26]:
# --- Load and filter BeetleBox dataset ---
df = pd.read_parquet('/home/cs21d002_eashaan/PhD/Objective1/data/processed/bug_reports_clean.parquet')
beetlebox_df = df[df['source_dataset'] == 'BeetleBox'].copy()
grouped = beetlebox_df.groupby('repo_name')
# Filter for the specific repo
subset_df = beetlebox_df[beetlebox_df['repo_name'] == 'localstack/localstack']

# Save to CSV
subset_df.to_csv('langchain_bug_reports.csv', index=False)